# Iniciando o Spark

In [2]:
## Bloco de codigo para instalar versao especifica dos pacotes
!pip install pyspark

In [4]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Trusted_base_pagamento") \
    .getOrCreate()

# Importando bibliotecas

In [5]:
import os
import pytz
import datetime
from datetime import datetime
#from pyspark.sql.types import *
#from pyspark.sql.functions import count, avg
#import sys
#import numpy as np
#from datetime import datetime
#from pyspark.sql import SQLContext
#from datetime import timedelta
#from datetime import date
#from dateutil.relativedelta import relativedelta
#from pyspark.sql.functions import udf, lpad, translate

# Funções auxiliares e variáveis

In [12]:
# Função de log
def log():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S') + " >>>"

# Timestamp de processamento (com hora/minuto/segundo)
agora = datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc = agora.strftime("%Y%m%d%H%M%S")

# Data da execução (AAAAmmdd)
PROCESS_DATE = datetime.now().strftime("%Y%m%d")

# Período de referência (AAAAmm)
REF_PERIOD = datetime.now().strftime("%Y%m")

#Alterar o path_padrao caso seus arquivos não estejam nesse mesmo caminho
path_padrao = "/content/gdrive/Othercomputers/Meu laptop"

# Buckets e nomes de saída
bucket_base = "base_pagamento"
bucket_raw = f"{path_padrao}/Database_raw/book_pagamento/dados_pagamento"
bucket_trusted = f"{path_padrao}/Database_trusted/book_pagamento/dados_pagamento"
bucket_control = f"{path_padrao}/Database_control/book_pagamento/dados_pagamento"
output_trusted = f"trusted_{bucket_base}"

# Prints para conferência
print("PROCESS_DATE:", PROCESS_DATE)
print("REF_PERIOD:", REF_PERIOD)
print("dthproc:", dthproc)
print("bucket_raw:", bucket_raw)
print("bucket_trusted:", bucket_trusted)
print("bucket_control:", bucket_control)

PROCESS_DATE: 20260210
REF_PERIOD: 202602
dthproc: 20260210104743
bucket_raw: /content/gdrive/Othercomputers/Meu laptop/Database_raw/book_pagamento/dados_pagamento
bucket_trusted: /content/gdrive/Othercomputers/Meu laptop/Database_trusted/book_pagamento/dados_pagamento
bucket_control: /content/gdrive/Othercomputers/Meu laptop/Database_control/book_pagamento/dados_pagamento


# Leitura dos dados na camada Raw

In [14]:
path_raw = bucket_raw

parquet_files = [path_raw for f in os.listdir(bucket_raw) if f.endswith('.parquet')]
df_raw_pagamento= spark.read.parquet(*parquet_files, header=True, inferSchema=True)

df_raw_pagamento.createOrReplaceTempView("raw_base_pagamento")

print(log(), "Registros na Raw:", df_raw_pagamento.count())
df_raw_pagamento.show(5, truncate=False)

2026-02-10 13:49:43 >>> Registros na Raw: 218296280
+-----------+------------------+---------+----------+------------------+---------------+--------------+-----------------+--------------+-------+-------------+------------------+--------------------+------------------+--------+-----------------+-------------------+---------------------+----------------+-----------------+-----------------+------------------+---------------------+--------------------+---------------------+------------------+-----------------+-------------------+----------------------+---------------------+-------------------------+----------------------------+-------------+-------------------+-------------------+-------------------+----------------------+-------------------+-------------------+-------------------+---------------------+----------------------+---------------------+-------------------------+-------------------+-------------------+----------------------+--------------------+------------------+---------------

# Processamento tipagem para camada Trusted

In [15]:
df_trusted_pagamento = spark.sql(f"""
SELECT
  '{dthproc}' AS ts_proc,
  '{dthproc}' AS ts_proc_partition,
  CAST(NUM_CPF AS STRING) AS NUM_CPF,
  TO_DATE(DAT_STATUS_FATURA, 'ddMMMyyyy:HH:mm:ss') AS DAT_STATUS_FATURA,
  CAST(CONTRATO AS STRING) AS CONTRATO,
  CAST(SEQ_FATURA AS INT) AS SEQ_FATURA,
  CAST(NUM_SUB_SEQ_FATURA AS INT) AS NUM_SUB_SEQ_FATURA,
  CAST(NUM_CREDITO_SEQ AS INT) AS NUM_CREDITO_SEQ,
  CAST(DW_TIPO_FATURA AS INT) AS DW_TIPO_FATURA,
  CAST(IND_STATUS_FATURA AS STRING) AS IND_STATUS_FATURA,
  CAST(DW_NUM_CLIENTE AS STRING) AS DW_NUM_CLIENTE,
  CAST(DW_AREA AS INT) AS DW_AREA,
  CAST(DW_UN_NEGOCIO AS INT) AS DW_UN_NEGOCIO,
  CAST(DW_FORMA_PAGAMENTO AS INT) AS DW_FORMA_PAGAMENTO,
  CAST(VAL_PAGAMENTO_FATURA AS FLOAT) AS VAL_PAGAMENTO_FATURA,
  TO_DATE(DAT_CRIACAO_DW, 'ddMMMyyyy:HH:mm:ss') AS DAT_CRIACAO_DW,
  TO_CHAR(TO_TIMESTAMP(DAT_CRIACAO_DW, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss') AS HR_CRIACAO_DW,
  CAST(DW_BANCO AS STRING) AS DW_BANCO,
  CAST(DW_TIPO_PAGAMENTO AS STRING) AS DW_TIPO_PAGAMENTO,
  CAST(NUM_BANCO_PAGAMENTO AS STRING) AS NUM_BANCO_PAGAMENTO,
  CAST(NUM_AGENCIA_PAGAMENTO AS STRING) AS NUM_AGENCIA_PAGAMENTO,
  CAST(NUM_CC_PAGAMENTO AS STRING) AS NUM_CC_PAGAMENTO,
  CAST(DW_MOTIVO_ESTORNO AS STRING) AS DW_MOTIVO_ESTORNO,
  CAST(VAL_DESCONTO_ITEM AS FLOAT) AS VAL_DESCONTO_ITEM,
  CAST(VAL_PAGAMENTO_ITEM AS FLOAT) AS VAL_PAGAMENTO_ITEM,
  CAST(VAL_JUROS_MULTAS_ITEM AS FLOAT) AS VAL_JUROS_MULTAS_ITEM,
  CAST(VAL_MULTA_EQUIP_ITEM AS FLOAT) AS VAL_MULTA_EQUIP_ITEM,
  CAST(VAL_MULTA_EQUIP_TOTAL AS FLOAT) AS VAL_MULTA_EQUIP_TOTAL,
  CAST(VAL_MULTA_FID_ITEM AS FLOAT) AS VAL_MULTA_FID_ITEM,
  CAST(COD_ORIGEM_NETUNO AS STRING) AS COD_ORIGEM_NETUNO,
  CAST(COD_CONTA_ATIVIDADE AS STRING) AS COD_CONTA_ATIVIDADE,
  CAST(SEQ_ENTIDADE_ATIVIDADE AS INT) AS SEQ_ENTIDADE_ATIVIDADE,
  TO_DATE(DAT_CRIACAO_ATIVIDADE, 'ddMMMyyyy:HH:mm:ss')                              AS DAT_CRIACAO_ATIVIDADE,
  TO_CHAR(TO_TIMESTAMP(DAT_CRIACAO_ATIVIDADE, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')     AS HR_CRIACAO_ATIVIDADE,
  TO_DATE(DAT_ATUALIZACAO_ATIVIDADE, 'ddMMMyyyy:HH:mm:ss')                          AS DAT_ATUALIZACAO_ATIVIDADE,
  TO_CHAR(TO_TIMESTAMP(DAT_ATUALIZACAO_ATIVIDADE, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss') AS HR_ATUALIZACAO_ATIVIDADE,
  CAST(COD_LOGIN_OPERADOR_ATIVIDADE AS STRING) AS COD_LOGIN_OPERADOR_ATIVIDADE,
  CAST(COD_ATIVIDADE AS STRING) AS COD_ATIVIDADE,
  CAST(COD_RAZAO_ATIVIDADE AS STRING) AS COD_RAZAO_ATIVIDADE,
  TO_DATE(DAT_BAIXA_ATIVIDADE, 'ddMMMyyyy:HH:mm:ss') AS DAT_BAIXA_ATIVIDADE,
  CAST(VAL_BAIXA_ATIVIDADE AS FLOAT) AS VAL_BAIXA_ATIVIDADE,
  TO_DATE(DAT_DEPOSITO_ATIVIDADE, 'ddMMMyyyy:HH:mm:ss') AS DAT_DEPOSITO_ATIVIDADE,
  CAST(COD_FUNDO_ATIVIDADE AS STRING) AS COD_FUNDO_ATIVIDADE,
  CAST(COD_BANCO_ATIVIDADE AS STRING) AS COD_BANCO_ATIVIDADE,
  CAST(NUM_CONTA_ATIVIDADE AS STRING) AS NUM_CONTA_ATIVIDADE,
  CAST(COD_AGENCIA_ATIVIDADE AS STRING) AS COD_AGENCIA_ATIVIDADE,
  CAST(SEQ_ENTIDADE_PAGAMENTO AS INT) AS SEQ_ENTIDADE_PAGAMENTO,
  TO_DATE(DAT_CRIACAO_PAGAMENTO, 'ddMMMyyyy:HH:mm:ss')                              AS DAT_CRIACAO_PAGAMENTO,
  TO_CHAR(TO_TIMESTAMP(DAT_CRIACAO_PAGAMENTO, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')     AS HR_CRIACAO_PAGAMENTO,
  TO_DATE(DAT_ATUALIZACAO_PAGAMENTO, 'ddMMMyyyy:HH:mm:ss')                          AS DAT_ATUALIZACAO_PAGAMENTO,
  TO_CHAR(TO_TIMESTAMP(DAT_ATUALIZACAO_PAGAMENTO, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss') AS HR_ATUALIZACAO_PAGAMENTO,
  CAST(COD_LOGIN_PAGAMENTO AS STRING) AS COD_LOGIN_PAGAMENTO,
  CAST(COD_FORMA_PAGAMENTO AS STRING) AS COD_FORMA_PAGAMENTO,
  CAST(VAL_ORIGINAL_PAGAMENTO AS FLOAT) AS VAL_ORIGINAL_PAGAMENTO,
  CAST(NUM_FATURA_PAGAMENTO AS STRING) AS NUM_FATURA_PAGAMENTO,
  CAST(COD_TIPO_PAGAMENTO AS STRING) AS COD_TIPO_PAGAMENTO,
  CAST(DSC_NOME_BANCO_PAGAMENTO AS STRING) AS DSC_NOME_BANCO_PAGAMENTO,
  CAST(SEQ_ARQUIVO_PAGAMENTO AS INT) AS SEQ_ARQUIVO_PAGAMENTO,
  CAST(NUM_PARCELA_PAGAMENTO AS INT) AS NUM_PARCELA_PAGAMENTO,
  CAST(NUM_AGRUPADOR_PAGAMENTO AS INT) AS NUM_AGRUPADOR_PAGAMENTO,
  CAST(DSC_PAGAMENTO AS STRING) AS DSC_PAGAMENTO,
  CAST(VAL_ATUAL_PAGAMENTO AS FLOAT) AS VAL_ATUAL_PAGAMENTO,
  CAST(COD_METODO_PAGAMENTO AS INT) AS COD_METODO_PAGAMENTO,
  CAST(IND_STATUS_PAGAMENTO AS STRING) AS IND_STATUS_PAGAMENTO,
  TO_DATE(DAT_STATUS_PAGAMENTO, 'ddMMMyyyy:HH:mm:ss') AS DAT_STATUS_PAGAMENTO,
  CAST(COD_ARQUIVO_PAGAMENTO AS STRING) AS COD_ARQUIVO_PAGAMENTO,
  CAST(COD_NETUNO_PAGAMENTO AS STRING) AS COD_NETUNO_PAGAMENTO,
  TO_DATE(DAT_CRIACAO_CREDITO, 'ddMMMyyyy:HH:mm:ss')                          AS DAT_CRIACAO_CREDITO,
  TO_CHAR(TO_TIMESTAMP(DAT_CRIACAO_CREDITO, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss') AS HR_CRIACAO_CREDITO,
  TO_DATE(DAT_ATUALIZACAO_CREDITO, 'ddMMMyyyy:HH:mm:ss')                      AS DAT_ATUALIZACAO_CREDITO,
  CAST(COD_LOGIN_CREDITO AS STRING) AS COD_LOGIN_CREDITO,
  CAST(VAL_PAGAMENTO_CREDITO AS FLOAT) AS VAL_PAGAMENTO_CREDITO,
  CAST(IND_TIPO_CREDITO AS STRING) AS IND_TIPO_CREDITO,
  CAST(SEQ_PAGAMENTO_CREDITO AS INT) AS SEQ_PAGAMENTO_CREDITO,
  CAST(SEQ_FATURA_CREDITO AS INT) AS SEQ_FATURA_CREDITO,
  CAST(COD_ALOCACAO_CREDITO AS STRING) AS COD_ALOCACAO_CREDITO,
  CAST(COD_DESALOCACAO_CREDITO AS STRING) AS COD_DESALOCACAO_CREDITO,
  CAST(SEQ_ENTIDADE_CREDITO AS INT) AS SEQ_ENTIDADE_CREDITO,
  CAST(COD_TIPO_FATURA AS STRING) AS COD_TIPO_FATURA,
  TO_DATE(DAT_ATIVIDADE_CREDITO, 'ddMMMyyyy:HH:mm:ss') AS DAT_ATIVIDADE_CREDITO,
  TO_DATE(DAT_VENCIMENTO_CREDITO, 'ddMMMyyyy:HH:mm:ss') AS DAT_VENCIMENTO_CREDITO
FROM raw_base_pagamento
""")

df_trusted_pagamento.createOrReplaceTempView("lake_base_pagamento")
#df_trusted_pagamento.cache()

print(log(), "Registros Trusted:", df_trusted_pagamento.count())
#df_trusted_pagamento.printSchema()
df_trusted_pagamento.show(5, truncate=False)

2026-02-10 13:50:23 >>> Registros Trusted: 218296280
+--------------+-----------------+-----------+-----------------+---------+----------+------------------+---------------+--------------+-----------------+--------------+-------+-------------+------------------+--------------------+--------------+-------------+--------+-----------------+-------------------+---------------------+----------------+-----------------+-----------------+------------------+---------------------+--------------------+---------------------+------------------+-----------------+-------------------+----------------------+---------------------+--------------------+-------------------------+------------------------+----------------------------+-------------+-------------------+-------------------+-------------------+----------------------+-------------------+-------------------+-------------------+---------------------+----------------------+---------------------+--------------------+-------------------------+--------

# Salvar na camada Trusted

In [ ]:
path_trusted = os.path.join(bucket_trusted, output_trusted)
df_trusted_pagamento.write \
    .partitionBy("ts_proc_partition") \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(path_trusted)

# Controle de carga

In [17]:
controle = spark.sql (f"""
    SELECT
        '{output_trusted}' AS name_file,
        ts_proc,
        count(*) as qtd_registros
    from lake_base_pagamento
    GROUP BY 1,2
""")

controle.createOrReplaceTempView("controle")
#controle.cache()

print(log(), "Registros controle:", controle.count())
controle.show(truncate=False)

2026-02-10 13:51:33 >>> Registros controle: 1
+----------------------+--------------+-------------+
|name_file             |ts_proc       |qtd_registros|
+----------------------+--------------+-------------+
|trusted_base_pagamento|20260210104743|218296280    |
+----------------------+--------------+-------------+



# Controle de processamento

In [ ]:
path_control = os.path.join(bucket_control, f'tb_controle_processamento_{bucket_base}_trusted')
print("Control path:", path_control)

controle.write \
  .mode('append') \
  .option('compression', 'snappy') \
  .parquet(path_control)